In [ ]:
# Instalar dependencias para las conexiones de datos
!pip install psycopg2-binary sqlalchemy pyodbc python-dotenv

   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.8 MB ? eta -:--:--
   ---------------------------------------- 2.8/2.8 MB 10.3 MB/s  0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 2.2/2.2 MB 23.4 MB/s  0:00:00

   ------------- -------------------------- 1/3 [greenlet]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------

In [5]:
# Importar librerías para conectarse y trabajar con dataframes
from sqlalchemy import create_engine
import pandas as pd
import pyodbc
from urllib.parse import quote_plus
import os
from dotenv import load_dotenv

# Cargar las variables de conexión desde el archivo .env
load_dotenv()

True

In [6]:
# Configurar la conexión a PostgreSQL de origen usando las variables del .env
usuario = os.getenv("PG_USUARIO")
password = os.getenv("PG_PASSWORD")
servidor = os.getenv("PG_SERVIDOR")
puerto = os.getenv("PG_PUERTO")
base_datos = os.getenv("PG_BASE_DATOS")

cadena = (
    f"postgresql+psycopg2://"
    f"{usuario}:{password}@"
    f"{servidor}:{puerto}/"
    f"{base_datos}"
)

engine_origen = create_engine(cadena)

try:
    with engine_origen.connect() as conexion:
        print("Conexión exitosa")
except Exception as error:
    print(type(error))
    print(error)

Conexión exitosa


In [14]:
# Obtener las categorías desde PostgreSQL
consulta_categoria = "SELECT * FROM tienda.categoria"
df_categoria = pd.read_sql_query(consulta_categoria, engine_origen)

In [ ]:
# Configurar la conexión al stage en SQL Server usando las variables del .env
servidor_sqlserver = os.getenv("SQLSERVER_SERVIDOR")
usuario_sqlserver = os.getenv("SQLSERVER_USUARIO")
password_sqlserver = os.getenv("SQLSERVER_PASSWORD")
base_datos_stage = os.getenv("SQLSERVER_BASE_DATOS")
esquema_stage = os.getenv("SQLSERVER_ESQUEMA")
driver_sqlserver = os.getenv("SQLSERVER_DRIVER")

cadena_sqlserver = (
    f"mssql+pyodbc://{usuario_sqlserver}:{quote_plus(password_sqlserver)}@"
    f"{servidor_sqlserver}/{base_datos_stage}?driver={quote_plus(driver_sqlserver)}"
)

engine_stage = create_engine(cadena_sqlserver)
try:
    with engine_stage.connect() as conexion:
        print("Conexión exitosa")
except Exception as error:
    print(type(error))
    print(error)

Conexión exitosa


In [ ]:
# Agregar el mes de creación y la fecha y hora de carga al dataframe
df_categoria["mes_creacion"] = pd.to_datetime(df_categoria["fecha_creacion"]).dt.month
df_categoria["fecha_carga"] = pd.Timestamp.now()

In [16]:
df_categoria.head()

,categoria_id,nombre,descripcion,activo,fecha_creacion,fecha_modificacion,fecha_carga
0,1,Figuras y coleccionables,"Figuras, estatuas y artículos de colección",True,2026-09-16 14:22:49.159577,2026-09-16 14:22:49.159577,2026-09-18 12:23:32.776219
1,2,Cómics y manga,Publicaciones impresas y novelas gráficas,True,2026-09-16 14:22:49.159577,2026-09-16 14:22:49.159577,2026-09-18 12:23:32.776219
2,3,Juegos de mesa,"Juegos familiares, de estrategia y rol",True,2026-09-16 14:22:49.159577,2026-09-16 14:22:49.159577,2026-09-18 12:23:32.776219
3,4,Ropa y accesorios,Prendas y accesorios temáticos,True,2026-09-16 14:22:49.159577,2026-09-16 14:22:49.159577,2026-09-18 12:23:32.776219
4,5,Hogar y decoración,Artículos decorativos para el hogar,True,2026-09-16 14:22:49.159577,2026-09-16 14:22:49.159577,2026-09-18 12:23:32.776219


In [ ]:
# Cargar las categorías en la tabla del stage
df_categoria.to_sql(
    name="categoria",
    con=engine_stage,
    schema=esquema_stage,
    if_exists="append",
    index=False
)

In [ ]:
# Verifico el contenido de la tabla en el stage
consulta_stage = f"SELECT * FROM {esquema_stage}.categoria"